# KAS-PPI: thực nghiệm đầy đủ (cập nhật 06/10/2026)

Notebook này chạy lại toàn bộ kết quả trong tài liệu *Chốt mô hình luận văn: KAS-PPI*:
1. Tái hiện đặc trưng KGsim2vec (bản Python) và biểu diễn SEEK, kèm kiểm tra khớp với file của tác giả.
2. Dự đoán 10-fold cho 3 view (KGsim2vec, SEEK, Hybrid) × RF/XGB, có và không có aspect `protein-containing complex`.
3. Giải thích bằng KAS và các baseline (LOO, FS0, TreeSHAP, LIME, thứ tự ngẫu nhiên), đo các tiêu chí VXAI.
4. Đo ảnh hưởng của aspect một phía (mục 3 của doc giải thích).

**Bước 3 có ba chế độ** (ô cấu hình ở mục 3):

| Chế độ | Nội dung | Số lượt | Số cặp |
|---|---|---|---|
| `quick` | Fold 1, 100 cặp mỗi cấu hình | 6 | 600 |
| `full` | Đúng như doc: 10 fold × 3 view × RF/XGB (fold 1: 400 cặp, fold 2–10: 200 cặp) + MLP ở fold 1–3 (200 cặp) | 69 | 15,000 |

Thời gian đo được: `full` mất khoảng 2.5 giờ trên máy 4 CPU chạy 4 lượt song song. Ước tính trên Colab miễn phí (2 CPU) khoảng 5 giờ. Kết quả từng lượt được ghi ra `rec_*.json`, nên nếu Colab ngắt giữa chừng, chạy lại ô thì các lượt đã xong sẽ được bỏ qua.

Chỉ cần CPU. Notebook tái hiện hai repo gốc (SML Java, bảng 1–4 của hai bài báo) là một file riêng: `Tai_hien_SEEK_KGsim2vec.ipynb`.

In [1]:
!git clone -q --depth 1 https://github.com/liseda-lab/ExplainablePPI.git
!git clone -q --depth 1 https://github.com/liseda-lab/seek.git
!mkdir -p godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/go-basic.zip -d godata && unzip -o -q ExplainablePPI/Experimental_Results/Data/goa_human.zip -d godata
!cd seek/PPI && unzip -o -q go.zip && unzip -o -q go_annotations.zip && unzip -o -q Embeddings/Emb_classes_maxdepth4_nwalks100.zip -d Embeddings && unzip -o -q Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.zip -d Embeddings
!pip -q install "numpy<2.0.0" "xgboost==1.5.2" "shap<=0.41.0" lime pyreadr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 67.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 24.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.9/362.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the p

## Mã nguồn
Mỗi ô ghi một file. `kgsim.py`: đặc trưng KGsim2vec (Resnik-max, IC Seco trong từng aspect). `seekrep.py`: biểu diễn SEEK (trung bình embedding RDF2Vec của các DCA, embedding lớp cố định của tác giả). `explain_v2.py`: bộ giải thích KAS và các baseline.

In [2]:
%%writefile kgsim.py
"""Python re-implementation of KGsim2vec features (Resnik-max, IC Seco computed inside each aspect subgraph)."""
import re, math, collections, pickle, os
OBO="http://purl.obolibrary.org/obo/GO_"
def parse_go(owl):
    parents=collections.defaultdict(set); label={}; cur=None
    pat_cls=re.compile(r'<owl:Class rdf:about="http://purl.obolibrary.org/obo/(GO_\d+)">')
    pat_sub=re.compile(r'<rdfs:subClassOf rdf:resource="http://purl.obolibrary.org/obo/(GO_\d+)"/>')
    pat_lab=re.compile(r'<rdfs:label[^>]*>([^<]*)</rdfs:label>')
    depth=0
    for line in open(owl,encoding="utf-8"):
        m=pat_cls.search(line)
        if m: cur=m.group(1); continue
        if cur is None: continue
        if "</owl:Class>" in line and line.strip()=="</owl:Class>": cur=None; continue
        m=pat_sub.search(line)
        if m: parents[cur].add(m.group(1)); continue
        m=pat_lab.search(line)
        if m and cur not in label: label[cur]=m.group(1)
    return parents,label
def load_gaf(gaf):
    ann=collections.defaultdict(set)
    for l in open(gaf):
        if l.startswith("!"): continue
        f=l.split("\t")
        if len(f)<5 or "NOT" in f[3]: continue
        ann[f[1]].add("GO_"+f[4].split(":")[1])
    return ann
class KG:
    def __init__(s,owl,gaf,aspects):
        s.parents,s.label=parse_go(owl)
        s.children=collections.defaultdict(set)
        for c,ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        s.ann=load_gaf(gaf)
        s.aspects=aspects
        s._anc={}
        s.sub=[]; s.ic=[]
        for a in aspects:
            members=s.desc(a)|{a}
            nprot=sum(1 for p,ts in s.ann.items() if ts & members)
            N=len(members)+nprot   # SML counts annotated instances as vertices of the reduced graph
            ic={}
            for c in members:
                d=len(s.desc(c))
                ic[c]=1-math.log(d+1)/math.log(N) if N>1 else 1.0
            s.sub.append(members); s.ic.append(ic)
    def desc(s,c,_cache={}):
        if c in _cache: return _cache[c]
        out=set(); st=[c]
        while st:
            x=st.pop()
            for y in s.children.get(x,()):
                if y not in out: out.add(y); st.append(y)
        _cache[c]=out; return out
    def anc(s,c):
        if c in s._anc: return s._anc[c]
        out={c}; st=[c]
        while st:
            x=st.pop()
            for y in s.parents.get(x,()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c]=out; return out
    def closure(s,terms):
        out=set()
        for t in terms: out|=s.anc(t)
        return out
    def features(s,t1,t2):
        """t1,t2: sets of direct annotation terms."""
        A1=s.closure(t1); A2=s.closure(t2)
        common=A1&A2
        v=[]
        for i,a in enumerate(s.aspects):
            m=s.sub[i]
            # protein needs >=1 direct annotation in aspect
            if not (t1 & m) or not (t2 & m): v.append(0.0); continue
            cm=common & m
            v.append(max((s.ic[i][c] for c in cm),default=0.0))
        return v


Writing kgsim.py


In [3]:
%%writefile seekrep.py
"""Python re-implementation of SEEK pair representations (Sousa et al., KR 2023):
pair vector = average RDF2Vec embedding of the disjoint common ancestors (DCA) of the two proteins' GO annotations.
Class embeddings are the authors' precomputed ones (fixed); only the DCA set changes when annotations are removed."""
import re, json, collections, numpy as np
from kgsim import parse_go
class SEEK:
    def __init__(s, owl, gaf, emb_file):
        s.parents, s.label = parse_go(owl)
        s.children = collections.defaultdict(set)
        for c, ps in s.parents.items():
            for p in ps: s.children[p].add(c)
        # SEEK's build_kg reads every GAF line (qualifier column ignored), protein id = column 2
        s.ann = collections.defaultdict(set)
        with open(gaf) as f:
            f.readline()
            for l in f:
                if l.startswith("!"): continue
                c = l.split("\t")
                s.ann[c[1]].add("GO_" + c[4].split(":")[1])
        txt = open(emb_file).read().replace("'", '"')
        raw = json.loads(re.sub(r'"http://purl.obolibrary.org/obo/', '"', txt))
        s.emb = {k: np.asarray(v, dtype=np.float64) for k, v in raw.items()}
        s.dim = len(next(iter(s.emb.values())))
        s._anc = {}
    def anc(s, c):
        if c in s._anc: return s._anc[c]
        out = {c}; st = [c]
        while st:
            x = st.pop()
            for y in s.parents.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        s._anc[c] = frozenset(out); return s._anc[c]
    def desc(s, c, _cache={}):
        if c in _cache: return _cache[c]
        out = set(); st = [c]
        while st:
            x = st.pop()
            for y in s.children.get(x, ()):
                if y not in out: out.add(y); st.append(y)
        _cache[c] = out; return out
    def dca(s, t1, t2):
        A1 = set().union(*[s.anc(t) for t in t1]) if t1 else set()
        A2 = set().union(*[s.anc(t) for t in t2]) if t2 else set()
        common = A1 & A2
        strict = set()
        for d in common: strict |= (s.anc(d) - {d})
        return [c for c in common - strict if c in s.emb]
    def rep(s, t1, t2):
        d = s.dca(t1, t2)
        if not d: return np.zeros(s.dim)
        return np.mean([s.emb[c] for c in d], 0)


Writing seekrep.py


In [4]:
%%writefile build_feats.py
import sys,time,numpy as np,pickle
sys.path.insert(0,".")
from kgsim import KG
R="ExplainablePPI/Experimental_Results/"
aspects=[l.split("\t")[0].split("/")[-1] for l in open(R+"Results/kgsim2vec_sas.txt") if l.strip()]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects)
rows=[l.rstrip("\n").split("\t") for l in open(R+"Results/ss_resnik_max_ICseco.txt")]
lab={}
for l in open(R+"Data/v11(score950).txt"):
    f=l.split(); lab[(f[0],f[1])]=int(f[2])
keys=[(r[0].replace("http://",""),r[1].replace("http://","")) for r in rows]
Xref=np.array([[float(x) for x in r[2:]] for r in rows])
t=time.time()
Xpy=np.array([kg.features(kg.ann[a],kg.ann[b]) for a,b in keys])
print("built",time.time()-t)
y=np.array([lab[k] for k in keys])
np.savez("feats.npz",Xref=Xref,Xpy=Xpy,y=y,keys=np.array(keys),aspects=np.array(aspects))
d=np.abs(Xref-Xpy)
print("exact pairs",(d.max(1)<1e-6).mean(),"cells within 1e-6",(d<1e-6).mean(),"within 0.05",(d<0.05).mean())
print("zero-pattern agreement",((Xref>0)==(Xpy>0)).mean())
cors=[np.corrcoef(Xref[:,j],Xpy[:,j])[0,1] for j in range(52) if Xref[:,j].std()>0 and Xpy[:,j].std()>0]
print("per-aspect pearson median %.4f min %.4f"%(np.median(cors),np.min(cors)))


Writing build_feats.py


In [5]:
%%writefile check_seek.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
t=time.time(); sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt"); print("load",time.time()-t, len(sk.emb))
ref={}
for l in open(S+"Embeddings/Emb_pair_maxdepth4_nwalks100_Avg_disjointcommonancestor.txt"):
    f=l.rstrip("\n").split("\t"); ref[(f[0],f[1])]=np.array(f[2:],float)
pairs=[l.split("\t")[:2] for l in open(S+"STRING_pairs.txt")]
rng=np.random.default_rng(0); idx=rng.choice(len(pairs),1000,replace=False)
err=[];t=time.time()
for i in idx:
    a,b=pairs[i]; v=sk.rep(sk.ann[a],sk.ann[b]); err.append(np.abs(v-ref[(a,b)]).max())
err=np.array(err); print("time/pair",(time.time()-t)/1000, "exact(1e-6)",(err<1e-6).mean(), "max",err.max(), "p99",np.quantile(err,.99))


Writing check_seek.py


In [6]:
%%writefile build_seek_feats.py
import numpy as np, time
from seekrep import SEEK
S="seek/PPI/"
sk=SEEK(S+"go.owl",S+"go_annotations.gaf",S+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
K=np.load("feats_kg.npz"); aspects=list(K["aspects"]); keys=[tuple(k) for k in K["keys"]]
roots=["GO_0008150","GO_0003674","GO_0005575"]
ch=set().union(*[sk.children[r] for r in roots])
print("aspects in SEEK GO:",len(ch),"overlap with KG aspects",len(ch&set(aspects)),"missing",set(aspects)-ch)
cx=sk.desc("GO_0032991")|{"GO_0032991"}
Xs=np.zeros((len(keys),sk.dim)); Xs51=np.zeros_like(Xs); t=time.time()
for i,(a,b) in enumerate(keys):
    t1,t2=sk.ann[a],sk.ann[b]
    Xs[i]=sk.rep(t1,t2); Xs51[i]=sk.rep(t1-cx,t2-cx)
print("built",time.time()-t, "rows changed by complex removal",(np.abs(Xs-Xs51).max(1)>0).mean())
np.savez("feats_seek.npz",Xseek=Xs,Xseek51=Xs51)


Writing build_seek_feats.py


In [7]:
%%writefile train_v2.py
"""10-fold prediction table for the three pipelines (repo partitions), with and without the circular aspect."""
import numpy as np, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from scipy.stats import wilcoxon
K=np.load("feats_kg.npz"); S=np.load("feats_seek.npz"); y=K["y"]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]
P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds=[[int(l) for l in open(P+("1(complete)" if r==1 else str(r))+".txt") if l.strip()] for r in range(1,11)]
feats={"KG_52":K["Xpy"],"KG_51":K["Xpy"][:,keep],"SEEK_52":S["Xseek"],"SEEK_51":S["Xseek51"],
       "HYB_52":np.hstack([K["Xpy"],S["Xseek"]]),"HYB_51":np.hstack([K["Xpy"][:,keep],S["Xseek51"]])}
out={}
for mname in ["RF100","XGB300"]:
    for fname,X in feats.items():
        f1s=[];aucs=[]
        for test in folds:
            m=np.zeros(len(y),bool); m[test]=True
            clf=RandomForestClassifier(100,n_jobs=-1,random_state=0) if mname=="RF100" else XGBClassifier(n_estimators=300,n_jobs=-1,random_state=0,eval_metric="logloss")
            clf.fit(X[~m],y[~m]); p=clf.predict_proba(X[m])[:,1]
            f1s.append(f1_score(y[m],(p>0.5).astype(int),average="weighted")); aucs.append(roc_auc_score(y[m],p))
        out[f"{fname}|{mname}"]=dict(median_waf=float(np.median(f1s)),iqr=float(np.percentile(f1s,75)-np.percentile(f1s,25)),median_auc=float(np.median(aucs)),folds=[round(x,4) for x in f1s],aucs=[round(x,4) for x in aucs])
        print(fname,mname,"WAF %.4f IQR %.4f AUC %.4f"%(np.median(f1s),out[f"{fname}|{mname}"]["iqr"],np.median(aucs)),flush=True)
    for a,b in [("HYB_51","KG_51"),("KG_51","SEEK_51"),("HYB_51","SEEK_51"),("SEEK_52","SEEK_51")]:
        fa=np.array(out[f"{a}|{mname}"]["folds"]); fb=np.array(out[f"{b}|{mname}"]["folds"])
        out[f"wilcoxon|{mname}|{a}_vs_{b}"]=dict(mean_diff=float((fa-fb).mean()),wins=int((fa>fb).sum()),p=float(wilcoxon(fa,fb).pvalue))
        print(mname,a,"vs",b,out[f"wilcoxon|{mname}|{a}_vs_{b}"],flush=True)
json.dump(out,open("results_prediction_v2.json","w"),indent=1)


Writing train_v2.py


In [8]:
%%writefile explain_v2.py
"""Direction D, full basic experiment: knowledge-space aspect Shapley (D) on three pipelines.
usage: python3 explain_v2.py PIPE MODEL FOLD N    PIPE in {kg,seek,hyb}, MODEL in {rf,xgb}
Players = GO aspects (51 direct children of the 3 GO roots, 'protein-containing complex' removed) in which
both proteins have >=1 direct annotation. Removing a player = deleting both proteins' annotations under that
aspect and recomputing the features through the pipeline (KGsim2vec: Resnik-max/IC-Seco; SEEK: DCA + mean
RDF2Vec embedding with the authors' fixed class embeddings)."""
import sys, time, json, itertools, math, random
import numpy as np

# Vá tương thích toàn diện cho shap <= 0.41 trên NumPy mới
for attr, target in [
    ("bool", bool),
    ("bool8", np.bool_),
    ("int", int),
    ("float", float),
]:
  if not hasattr(np, attr):
    setattr(np, attr, target)

sys.path.insert(0,"."); sys.path.insert(0,"ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap
from lime import lime_tabular

PIPE,MODEL,FOLD,N_PAIRS=sys.argv[1],sys.argv[2],int(sys.argv[3]),int(sys.argv[4])
M_PERM=100; EXACT_MAX=8; TOPK=5
rng=np.random.default_rng(FOLD); random.seed(FOLD)

K=np.load("feats_kg.npz"); y=K["y"]; keys=[tuple(k) for k in K["keys"]]; aspects=list(K["aspects"])
ci=aspects.index("GO_0032991"); keep=[j for j in range(52) if j!=ci]; A=[aspects[j] for j in keep]
SF=np.load("feats_seek.npz")
X={"kg":K["Xpy"][:,keep],"seek":SF["Xseek51"],"hyb":np.hstack([K["Xpy"][:,keep],SF["Xseek51"]])}[PIPE]
kg=KG("godata/go-basic.owl","godata/goa_human.gaf",aspects); sub_kg=[kg.sub[j] for j in keep]
S_="seek/PPI/"
sk=SEEK(S_+"go.owl",S_+"go_annotations.gaf",S_+"Embeddings/Emb_classes_maxdepth4_nwalks100.txt")
CX=sk.desc("GO_0032991")|{"GO_0032991"}
sub_sk=[sk.desc(a)|{a} for a in A]

P="ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
test=[int(l) for l in open(P+("1(complete)" if FOLD==1 else str(FOLD))+".txt") if l.strip()]
m=np.zeros(len(y),bool); m[test]=True
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
clf=(RandomForestClassifier(100,n_jobs=1,random_state=0) if MODEL=="rf"
     else XGBClassifier(n_estimators=300,n_jobs=1,random_state=0,eval_metric="logloss") if MODEL=="xgb"
     else make_pipeline(StandardScaler(),MLPClassifier((100,),max_iter=300,early_stopping=True,random_state=0))).fit(X[~m],y[~m])
def proba(Z): return clf.predict_proba(np.atleast_2d(Z))[:,1].astype(np.float64)

# state = (kg annotations p1, kg annotations p2, seek annotations p1, seek annotations p2)
def feats(st):
    k1,k2,s1,s2=st; parts=[]
    if PIPE in("kg","hyb"):
        v=kg.features(k1,k2); parts.append([v[j] for j in keep])
    if PIPE in("seek","hyb"): parts.append(list(sk.rep(s1,s2)))
    return np.concatenate(parts)
def players_of(st):
    k1,k2,s1,s2=st; out=[]
    for i in range(len(A)):
        a=(PIPE!="seek") and bool(k1&sub_kg[i]) and bool(k2&sub_kg[i])
        b=(PIPE!="kg") and bool(s1&sub_sk[i]) and bool(s2&sub_sk[i])
        if a or b: out.append(i)
    return out
def remove(st,S):
    if not S: return st
    rk=set().union(*[sub_kg[i] for i in S]); rs=set().union(*[sub_sk[i] for i in S])
    k1,k2,s1,s2=st; return (k1-rk,k2-rk,s1-rs,s2-rs)

def shapley(value_batch, players, seed):
    k=len(players); phi=np.zeros(len(A))
    if k==0: return phi
    if k<=EXACT_MAX:
        subsets=[frozenset(c) for r in range(k+1) for c in itertools.combinations(players,r)]
        vals=dict(zip(subsets,value_batch(subsets)))
        for i in players:
            others=[p for p in players if p!=i]
            for r in range(k):
                w=math.factorial(r)*math.factorial(k-r-1)/math.factorial(k)
                for c in itertools.combinations(others,r):
                    S=frozenset(c); phi[i]+=w*(vals[S]-vals[S|{i}])
        return phi
    r=random.Random(seed); perms=[r.sample(players,k) for _ in range(M_PERM)]
    subsets=set()
    for p in perms:
        for t in range(k+1): subsets.add(frozenset(p[:t]))
    subsets=list(subsets); vals=dict(zip(subsets,value_batch(subsets)))
    for p in perms:
        for t,i in enumerate(p): phi[i]+=vals[frozenset(p[:t])]-vals[frozenset(p[:t+1])]
    return phi/M_PERM

def explain_D(st,seed=0):
    pl=players_of(st)
    return shapley(lambda subs: proba(np.array([feats(remove(st,S)) for S in subs])),pl,seed)
def explain_LOO(st,seed=0):   # leave-one-aspect-out occlusion in knowledge space (aspect-level analogue of SEEK 'necessary')
    pl=players_of(st); phi=np.zeros(len(A))
    if not pl: return phi
    v=proba(np.array([feats(st)]+[feats(remove(st,[i])) for i in pl]))
    for j,i in enumerate(pl): phi[i]=v[0]-v[j+1]
    return phi
def explain_FS0(st,seed=0):   # KGsim2vec only: same players and sampler, feature set to 0 instead of recomputed
    pl=players_of(st); x=feats(st)
    def vb(subs):
        Z=np.repeat(x[None],len(subs),0)
        for r,S in enumerate(subs): Z[r,list(S)]=0
        return proba(Z)
    return shapley(vb,pl,seed)
bg=X[~m][rng.choice((~m).sum(),100,replace=False)]
tree=shap.TreeExplainer(clf,data=bg,feature_perturbation="interventional") if MODEL!="mlp" else None
def explain_TS(st,seed=0):    # kg: per aspect feature; hyb: only the 51 named KGsim2vec columns map to aspects
    x=feats(st); sv=tree.shap_values(x[None],check_additivity=False)
    sv=sv[1] if isinstance(sv,list) else sv
    sv=np.array(sv); sv=sv[0,:,1] if sv.ndim==3 else sv[0]
    return np.asarray(sv[:len(A)],dtype=np.float64)
if PIPE=="kg":
    lime_exp=lime_tabular.LimeTabularExplainer(X[~m][:2000],feature_names=A,class_names=["0","1"],discretize_continuous=True,random_state=0)
def explain_LIME(st,seed=0):
    x=feats(st); lime_exp.random_state=np.random.RandomState(seed)
    e=lime_exp.explain_instance(x,lambda Z: clf.predict_proba(Z).astype(np.float64),num_features=10,num_samples=2000,labels=(1,))
    phi=np.zeros(len(A))
    for j,w in e.as_map()[1]: phi[j]=w
    return phi
METHODS={"D":explain_D,"LOO":explain_LOO}
if PIPE=="kg": METHODS.update({"FS0":explain_FS0,"TreeSHAP":explain_TS,"LIME":explain_LIME})
if PIPE=="hyb": METHODS.update({"TreeSHAP_KGpart":explain_TS})
if MODEL=="mlp": METHODS={k:v for k,v in METHODS.items() if not k.startswith("TreeSHAP")}
DETERMINISTIC={"LOO","TreeSHAP","TreeSHAP_KGpart"}

def ranking(phi,pred):
    s=phi if pred==1 else -phi
    return [int(j) for j in np.argsort(-s) if s[j]>0]
def deletion_ks(st,order,pred):
    Z=np.array([feats(remove(st,order[:k])) for k in range(1,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def insertion_ks(st,order,pred,pl):  # start with every player removed, add back the top-k
    Z=np.array([feats(remove(st,[i for i in pl if i not in order[:k]])) for k in range(0,TOPK+1)]); p=proba(Z)
    return list(p if pred==1 else 1-p)
def eff_size(phi):
    a=np.sort(np.abs(phi))[::-1]; tot=a.sum()
    return int(np.searchsorted(np.cumsum(a)/tot,0.8)+1) if tot>0 else 0
def jacc(a,b,k=3):
    A_,B_=set(a[:k]),set(b[:k]); return len(A_&B_)/len(A_|B_) if (A_|B_) else 1.0

pos=[i for i in test if y[i]==1]; neg=[i for i in test if y[i]==0]
idx=list(rng.choice(pos,N_PAIRS//2,replace=False))+list(rng.choice(neg,N_PAIRS//2,replace=False))
rec=[]; t0=time.time()
for n,i in enumerate(idx):
    p1,p2=keys[i]
    st=(set(kg.ann[p1]),set(kg.ann[p2]),set(sk.ann[p1])-CX,set(sk.ann[p2])-CX)
    p=proba(feats(st))[0]; pred=int(p>0.5); conf=p if pred else 1-p
    pl=players_of(st)
    r=dict(i=int(i),pair=f"{p1}-{p2}",y=int(y[i]),pred=pred,conf=float(conf),k=len(pl))
    # continuity: drop ONE random direct annotation (same GO term in both annotation sets) lying in a player aspect
    cand=sorted({(w,t) for w in (0,1) for t in (st[w]|st[w+2]) if any(t in sub_kg[a] or t in sub_sk[a] for a in pl)})
    pert=None
    if cand:
        w,t=cand[rng.integers(len(cand))]; q=list(st); q[w]=q[w]-{t}; q[w+2]=q[w+2]-{t}; q=tuple(q)
        if int(proba(feats(q))[0]>0.5)==pred: pert=q
    ro=[int(v) for v in rng.permutation(pl)] if pl else []
    r["del_random"]=deletion_ks(st,ro,pred) if ro else None
    r["ins_random"]=insertion_ks(st,ro,pred,pl) if ro else None
    for name,f in METHODS.items():
        ts_=time.time(); phi=f(st,0); r[f"time_{name}"]=time.time()-ts_
        order=ranking(phi,pred)
        r[f"cov_{name}"]=int(len(order)>0); r[f"size_{name}"]=eff_size(phi)
        r[f"del_{name}"]=deletion_ks(st,order,pred) if order else None
        r[f"ins_{name}"]=insertion_ks(st,order,pred,pl) if order else None
        r[f"top3_{name}"]=[A[j] for j in order[:3]]
        r[f"phi_{name}"]={A[j]:round(float(phi[j]),5) for j in np.nonzero(phi)[0]}
        if name in DETERMINISTIC or (name=="D" and len(pl)<=EXACT_MAX):
            r[f"cons_jacc_{name}"]=1.0; r[f"cons_rho_{name}"]=1.0
        else:
            phi2=f(st,1); o2=ranking(phi2,pred); r[f"cons_jacc_{name}"]=jacc(order,o2)
            r[f"cons_rho_{name}"]=float(spearmanr(phi,phi2).correlation) if np.std(phi)>0 and np.std(phi2)>0 else None
        r[f"cont_{name}"]=jacc(order,ranking(f(pert,0),pred)) if pert is not None else None
    rec.append(r)
    if n%25==0: print(PIPE,MODEL,FOLD,n,round(time.time()-t0,1),"s k=",len(pl),flush=True)
json.dump(dict(pipe=PIPE,model=MODEL,fold=FOLD,aspects=A,labels={a:kg.label.get(a,a) for a in A},records=rec),open(f"rec_{PIPE}_{MODEL}_f{FOLD}_{N_PAIRS}.json","w"))
print("done",time.time()-t0)

Writing explain_v2.py


In [9]:
%%writefile analyze_v2.py
"""Summaries for explain_v2 runs: per run and pooled per (pipeline, model) over folds."""
import json, glob, collections, numpy as np
from scipy.stats import wilcoxon
rng=np.random.default_rng(0)
def curve(r,key,lo=0):
    v=r.get(key); return float(np.mean(v[lo:])) if v else None
def ci(a):
    b=[np.mean(rng.choice(a,len(a))) for _ in range(2000)]; return [float(x) for x in np.percentile(b,[2.5,97.5])]
def summarize(R,methods):
    out={"n":len(R),"acc":float(np.mean([r["pred"]==r["y"] for r in R])),
         "k_median":float(np.median([r["k"] for r in R])),"k_zero":int(sum(r["k"]==0 for r in R))}
    dels={};inss={}
    for m in methods+["random"]:
        # empty explanation: nothing is removed (deletion stays at conf), nothing is added back (insertion stays at base)
        dels[m]=np.array([curve(r,f"del_{m}") if r.get(f"del_{m}") else r["conf"] for r in R])
        inss[m]=np.array([curve(r,f"ins_{m}",1) if r.get(f"ins_{m}") else (r["ins_random"][0] if r.get("ins_random") else r["conf"]) for r in R])
        out[f"del_{m}"]=dict(mean=float(dels[m].mean()),ci95=ci(dels[m]))
        out[f"ins_{m}"]=dict(mean=float(inss[m].mean()),ci95=ci(inss[m]))
    for m in methods[1:]+["random"]:
        for nm,arr,sign in [("del",dels,1),("ins",inss,-1)]:
            a,b=arr["D"],arr[m]; d=(a-b)*sign
            try: p=float(wilcoxon(a,b).pvalue)
            except ValueError: p=1.0
            out[f"wil_{nm}_D_vs_{m}"]=dict(diff=float((a-b).mean()),D_better=float((d<0).mean()),ties=float((d==0).mean()),p=p)
    for m in methods:
        out[f"pars_{m}"]=float(np.mean([r[f"size_{m}"] for r in R]))
        out[f"cov_{m}"]=float(np.mean([r[f"cov_{m}"] for r in R]))
        out[f"cons_{m}"]=float(np.mean([r[f"cons_jacc_{m}"] for r in R]))
        c=[r[f"cont_{m}"] for r in R if r[f"cont_{m}"] is not None]; out[f"cont_{m}"]=float(np.mean(c)); out["cont_n"]=len(c)
        out[f"time_{m}"]=float(np.median([r[f"time_{m}"] for r in R]))
    if "LOO" in methods: out["D_vs_LOO_top3_differs"]=float(np.mean([r["top3_D"]!=r["top3_LOO"] for r in R]))
    if "FS0" in methods: out["D_vs_FS0_top3_differs"]=float(np.mean([r["top3_D"]!=r["top3_FS0"] for r in R]))
    return out
runs={}; pooled=collections.defaultdict(list); labels={}
for f in sorted(glob.glob("rec_*_f*_*.json")):
    J=json.load(open(f)); labels=J["labels"]; R=J["records"]
    methods=[k[5:] for k in R[0] if k.startswith("time_")]
    runs[f]=summarize(R,methods); pooled[(J["pipe"],J["model"])]+=R
allout={"runs":runs,"pooled":{}}
for (p,mo),R in pooled.items():
    methods=[k[5:] for k in R[0] if k.startswith("time_")]
    s=summarize(R,methods); allout["pooled"][f"{p}|{mo}"]=s
    # top aspects in D explanations of correctly predicted positives (input for a plausibility case study)
    cnt=collections.Counter(a for r in R if r["y"]==1 and r["pred"]==1 for a in r["top3_D"][:1])
    s["top1_aspect_TP"]=[(labels.get(a,a),c) for a,c in cnt.most_common(8)]
json.dump(allout,open("summary_v2.json","w"),indent=1)
for key,s in allout["pooled"].items():
    print("=====",key,"n",s["n"],"acc %.3f"%s["acc"],"k_med",s["k_median"])
    ms=[k[4:] for k in s if k.startswith("del_")]
    for m in ms: print("  %-16s del %.3f  ins %.3f"%(m,s[f"del_{m}"]["mean"],s[f"ins_{m}"]["mean"]),
                       "" if m=="random" else "pars %.2f cov %.3f cons %.3f cont %.3f t %.3f"%(s[f"pars_{m}"],s[f"cov_{m}"],s[f"cons_{m}"],s[f"cont_{m}"],s[f"time_{m}"]))
    for k,v in s.items():
        if k.startswith("wil_"): print("  ",k,{a:(round(b,4) if isinstance(b,float) else b) for a,b in v.items()})

Writing analyze_v2.py


## 1. Đặc trưng và kiểm tra tái hiện

In [10]:
!python build_feats.py && cp feats.npz feats_kg.npz
!python check_seek.py
!python build_seek_feats.py

built 6.8406407833099365
exact pairs 0.03497942386831276 cells within 1e-6 0.9616685921096001 within 0.05 0.9989548761026424
zero-pattern agreement 0.9993110014587678
per-aspect pearson median 1.0000 min 0.7884
load 3.5643486976623535 50422
time/pair 0.0002115778923034668 exact(1e-6) 1.0 max 0.0 p99 0.0
aspects in SEEK GO: 52 overlap with KG aspects 52 missing set()
built 11.15592074394226 rows changed by complex removal 0.4109711085656103


## 2. Dự đoán 10-fold (3 view × RF/XGB, 51 và 52 aspect)

In [11]:
!python train_v2.py

/usr/local/lib/python3.13/dist-packages/xgboost/compat.py:106: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
KG_52 RF100 WAF 0.9306 IQR 0.0047 AUC 0.9780
KG_51 RF100 WAF 0.9246 IQR 0.0043 AUC 0.9746
SEEK_52 RF100 WAF 0.9073 IQR 0.0077 AUC 0.9668
SEEK_51 RF100 WAF 0.9039 IQR 0.0056 AUC 0.9646
HYB_52 RF100 WAF 0.9283 IQR 0.0056 AUC 0.9770
HYB_51 RF100 WAF 0.9247 IQR 0.0039 AUC 0.9753
RF100 HYB_51 vs KG_51 {'mean_diff': -0.0009100000000000108, 'wins': 4, 'p': 0.556640625}
RF100 KG_51 vs SEEK_51 {'mean_diff': 0.02128000000000002, 'wins': 10, 'p': 0.001953125}
RF100 HYB_51 vs SEEK_51 {'mean_diff': 0.02037000000000001, 'wins': 10, 'p': 0.001953125}
RF100 SEEK_52 vs SEEK_51 {'mean_diff': 0.004289999999999983, 'wins': 10, 'p': 0.001953125}
/usr/local/lib/python3.13/dist-packages

## 3. Giải thích và tiêu chí VXAI
Chọn `MODE`. Các lượt chạy song song theo số CPU. Lượt đã có file `rec_*.json` sẽ được bỏ qua.

In [12]:
MODE = 'quick'   # 'quick' | 'full'
import subprocess, os
from concurrent.futures import ThreadPoolExecutor
if MODE == 'quick':
    jobs = [(p, m, 1, 100) for p in ['kg', 'seek', 'hyb'] for m in ['rf', 'xgb']]
else:
    jobs = [(p, m, f, 400 if f == 1 else 200) for f in range(1, 11) for p in ['kg', 'seek', 'hyb'] for m in ['rf', 'xgb']]
    jobs += [(p, 'mlp', f, 200) for f in [1, 2, 3] for p in ['kg', 'seek', 'hyb']]
def run(j):
    p, m, f, n = j
    if os.path.exists(f'rec_{p}_{m}_f{f}_{n}.json'): return f'{p} {m} f{f}: đã có, bỏ qua'
    r = subprocess.run(['python', 'explain_v2.py', p, m, str(f), str(n)], capture_output=True, text=True)
    return f'{p} {m} f{f}: ' + ('xong' if r.returncode == 0 else 'LỖI\n' + r.stderr[-1500:])
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    for msg in ex.map(run, jobs): print(msg, flush=True)

kg rf f1: xong
kg xgb f1: xong
seek rf f1: xong
seek xgb f1: xong
hyb rf f1: xong
hyb xgb f1: xong


In [13]:
!python analyze_v2.py

===== hyb|rf n 100 acc 0.920 k_med 5.0
  D                del 0.492  ins 0.847 pars 2.97 cov 0.990 cons 0.990 cont 0.948 t 0.063
  LOO              del 0.568  ins 0.841 pars 2.71 cov 1.000 cons 1.000 cont 0.889 t 0.010
  TreeSHAP_KGpart  del 0.618  ins 0.792 pars 3.04 cov 1.000 cons 1.000 cont 0.955 t 0.193
  random           del 0.659  ins 0.724 
   wil_del_D_vs_LOO {'diff': -0.0761, 'D_better': 0.64, 'ties': 0.13, 'p': 0.0}
   wil_ins_D_vs_LOO {'diff': 0.0069, 'D_better': 0.4, 'ties': 0.16, 'p': 0.4645}
   wil_del_D_vs_TreeSHAP_KGpart {'diff': -0.1263, 'D_better': 0.93, 'ties': 0.0, 'p': 0.0}
   wil_ins_D_vs_TreeSHAP_KGpart {'diff': 0.0557, 'D_better': 0.69, 'ties': 0.02, 'p': 0.0}
   wil_del_D_vs_random {'diff': -0.1667, 'D_better': 0.9, 'ties': 0.01, 'p': 0.0}
   wil_ins_D_vs_random {'diff': 0.1235, 'D_better': 0.87, 'ties': 0.03, 'p': 0.0}
===== hyb|xgb n 100 acc 0.960 k_med 5.0
  D                del 0.503  ins 0.943 pars 3.15 cov 1.000 cons 0.995 cont 0.908 t 0.050
  LOO        

## 4. Aspect một phía
Với 200 cặp (fold 1, RF): bỏ từng aspect mà chỉ một protein có chú giải, rồi đo thay đổi xác suất; so với player.

In [14]:
%%writefile onesided.py
"""How much do aspects annotated on only ONE protein (non-players) move the prediction when removed?"""
import sys, numpy as np, json

# Vá tương thích toàn diện cho shap <= 0.41 trên NumPy mới
for attr, target in [
    ("bool", bool),
    ("bool8", np.bool_),
    ("int", int),
    ("float", float),
]:
  if not hasattr(np, attr):
    setattr(np, attr, target)

sys.argv=["x",sys.argv[1],"rf","1","1"]
src=open("explain_v2.py").read().split("pos=[i for i in test")[0]   # reuse setup (data, model, feats, remove)
exec(src)
PIPE_=PIPE
pos=[i for i in test if y[i]==1]; neg=[i for i in test if y[i]==0]
r=np.random.default_rng(7); idx=list(r.choice(pos,100,replace=False))+list(r.choice(neg,100,replace=False))
d_pl=[]; d_one=[]; n_one=[]; flips_one=0; flips_pl=0
for i in idx:
    p1,p2=keys[i]; st=(set(kg.ann[p1]),set(kg.ann[p2]),set(sk.ann[p1])-CX,set(sk.ann[p2])-CX)
    p0=proba(feats(st))[0]; pl=set(players_of(st))
    one=[]
    for a in range(len(A)):
        if a in pl: continue
        if PIPE_=="kg": h1,h2=bool(st[0]&sub_kg[a]),bool(st[1]&sub_kg[a])
        else: h1,h2=bool(st[2]&sub_sk[a]),bool(st[3]&sub_sk[a])
        if h1!=h2: one.append(a)
    n_one.append(len(one))
    for a in one:
        p=proba(feats(remove(st,[a])))[0]; d_one.append(abs(p-p0)); flips_one+=int((p>0.5)!=(p0>0.5))
    for a in pl:
        p=proba(feats(remove(st,[a])))[0]; d_pl.append(abs(p-p0)); flips_pl+=int((p>0.5)!=(p0>0.5))
d_one=np.array(d_one); d_pl=np.array(d_pl)
print(json.dumps(dict(pipe=PIPE_,pairs=200,one_sided_per_pair=float(np.mean(n_one)),players_per_pair=float(len(d_pl)/200),
  mean_abs_dp_one=float(d_one.mean()),frac_one_zero=float((d_one<1e-9).mean()),frac_one_gt005=float((d_one>0.05).mean()),flips_one=flips_one,
  mean_abs_dp_player=float(d_pl.mean()),frac_pl_gt005=float((d_pl>0.05).mean()),flips_pl=flips_pl)))

Writing onesided.py


In [15]:
!python onesided.py kg
!python onesided.py seek

/content/onesided.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, attr):
/content/onesided.py:11: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not hasattr(np, attr):
<string>:17: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
/usr/local/lib/python3.13/dist-packages/xgboost/compat.py:106: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
{"pipe": "kg", "pairs": 200, "one_sided_per_pair": 4.65, "players_per_pair": 5.24, "mean_abs_dp_one": 0.005474979660920307, "frac_one_zero": 0.921505376344086, "frac_one_gt005": 0.04086021505376344, "flips_one": 1, "mean_abs_dp_player": 0.1313683314343906, "frac_pl_gt005": 0

## 5. ROAR và Plausibility định lượng (bổ sung 06/10/2026)
Cần `rec_*.json` của MODE `'full'` (2,200 cặp mỗi cấu hình). Nếu chưa chạy full, tải thư mục `records/` sẵn có lên rồi chép các file vào thư mục làm việc.
- `roar.py`: bỏ top-k aspect của từng cặp, huấn luyện lại, đo accuracy (5-fold × 3).
- `roar_mask.py`: kiểm tra rò rỉ mask.
- `get_msigdb.py`: lấy Reactome và KEGG từ MSigDB 7.5.1.
- `plaus.py`: hit@1 / NDCG@3 so với aspect được làm giàu trong pathway chung.
- `case_study.py`: bảng 12 cặp.
Báo cáo: `roar_plausibility/BAO_CAO_ROAR_PLAUSIBILITY.md`.

In [16]:
%%writefile roar.py
"""ROAR (RemOve And Retrain, Hooker et al. 2019) for the aspect explanations already stored in rec_{PIPE}_{MODEL}_f*.json.
usage: python3 roar.py PIPE MODEL      PIPE in {kg,seek,hyb}, MODEL in {rf,xgb}
Pool = every explained pair of that configuration (each pair was explained by the model of the fold where it is a TEST pair,
so explanations never come from a model that saw the pair). For each method and k in KS, each pair loses its own top-k
players; a fresh model of the same class is retrained and evaluated on the modified pool with repeated stratified 5-fold CV
(same splits for every method). A larger accuracy drop = the ranking removed more of the information the task needs.
Ranking: all players of the pair sorted by phi signed toward the original prediction (ties random), so every method removes
exactly min(k, #players) aspects. Removal operators:
  ks   = knowledge-space removal (delete both proteins' annotations under the aspect, recompute features)  [all pipes]
  zero = set the aspect's feature column to 0 (KGsim2vec only)"""
import sys, glob, json, time, random
import numpy as np
sys.path.insert(0, "."); sys.path.insert(0, "ExplainablePPI")
from kgsim import KG
from seekrep import SEEK
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

PIPE, MODEL = sys.argv[1], sys.argv[2]
KS = [1, 2, 3]; REPEATS = 3

K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"]); y_all = K["y"]
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub_kg = [kg.sub[j] for j in keep]
S_ = "seek/PPI/"
sk = SEEK(S_ + "go.owl", S_ + "go_annotations.gaf", S_ + "Embeddings/Emb_classes_maxdepth4_nwalks100.txt") if PIPE != "kg" else None
CX = (sk.desc("GO_0032991") | {"GO_0032991"}) if sk else set()
sub_sk = [sk.desc(a) | {a} for a in A] if sk else [set() for _ in A]

def feats(st):
    k1, k2, s1, s2 = st; parts = []
    if PIPE in ("kg", "hyb"):
        v = kg.features(k1, k2); parts.append([v[j] for j in keep])
    if PIPE in ("seek", "hyb"): parts.append(list(sk.rep(s1, s2)))
    return np.concatenate(parts)
def players_of(st):
    k1, k2, s1, s2 = st; out = []
    for i in range(len(A)):
        a = (PIPE != "seek") and bool(k1 & sub_kg[i]) and bool(k2 & sub_kg[i])
        b = (PIPE != "kg") and bool(s1 & sub_sk[i]) and bool(s2 & sub_sk[i])
        if a or b: out.append(i)
    return out
def remove(st, S):
    if not S: return st
    rk = set().union(*[sub_kg[i] for i in S]); rs = set().union(*[sub_sk[i] for i in S])
    k1, k2, s1, s2 = st; return (k1 - rk, k2 - rk, s1 - rs, s2 - rs)

recs = []
for f in sorted(glob.glob(f"rec_{PIPE}_{MODEL}_f*_*.json")):
    recs += json.load(open(f))["records"]
methods = [k[4:] for k in recs[0] if k.startswith("phi_")] + ["random"]
print(PIPE, MODEL, "pairs", len(recs), "methods", methods, flush=True)
aidx = {a: j for j, a in enumerate(A)}

rows = {m: {"ks": {k: [] for k in KS}, "zero": {k: [] for k in KS}} for m in methods}
base = []; y = []; npl = []
t0 = time.time()
for n, r in enumerate(recs):
    p1, p2 = keys[r["i"]]
    st = (set(kg.ann[p1]), set(kg.ann[p2]),
          (set(sk.ann[p1]) - CX) if sk else set(), (set(sk.ann[p2]) - CX) if sk else set())
    x0 = feats(st); base.append(x0); y.append(r["y"])
    pl = players_of(st); npl.append(len(pl))
    sgn = 1 if r["pred"] == 1 else -1
    rr = random.Random(r["i"])
    for m in methods:
        if m == "random":
            order = rr.sample(pl, len(pl))
        else:
            phi = r[f"phi_{m}"]; tieb = {i: rr.random() for i in pl}
            order = sorted(pl, key=lambda i: (-sgn * phi.get(A[i], 0.0), tieb[i]))
        for k in KS:
            rows[m]["ks"][k].append(feats(remove(st, order[:k])))
            if PIPE == "kg":
                z = x0.copy(); z[order[:k]] = 0; rows[m]["zero"][k].append(z)
    if n % 500 == 0: print(n, round(time.time() - t0, 1), "s", flush=True)
y = np.array(y); base = np.array(base)

def make():
    return (RandomForestClassifier(100, n_jobs=4, random_state=0) if MODEL == "rf"
            else XGBClassifier(n_estimators=300, n_jobs=4, random_state=0, eval_metric="logloss"))
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=REPEATS, random_state=0).split(base, y))
def cv(X):
    acc, f1 = [], []
    for tr, te in splits:
        c = make().fit(X[tr], y[tr]); p = c.predict(X[te])
        acc.append(accuracy_score(y[te], p)); f1.append(f1_score(y[te], p))
    return acc, f1
out = dict(pipe=PIPE, model=MODEL, n_pairs=len(y), median_players=float(np.median(npl)), ks=KS,
           n_splits=len(splits), methods=methods, base={}, res={})
a, f = cv(base); out["base"] = dict(acc=a, f1=f)
print("base acc %.4f f1 %.4f" % (np.mean(a), np.mean(f)), flush=True)
for op in (["ks", "zero"] if PIPE == "kg" else ["ks"]):
    for m in methods:
        for k in KS:
            a, f = cv(np.array(rows[m][op][k]))
            out["res"][f"{op}|{m}|{k}"] = dict(acc=a, f1=f)
            print(op, m, k, "acc %.4f f1 %.4f" % (np.mean(a), np.mean(f)), round(time.time() - t0), "s", flush=True)
json.dump(out, open(f"roar_{PIPE}_{MODEL}.json", "w"))
print("done", time.time() - t0)


Writing roar.py


In [17]:
%%writefile roar_mask.py
"""ROAR leakage diagnostic (cf. ROAD, Rong et al. 2022): how well can the label be predicted from WHICH aspects were removed
alone? Features = removed-aspect indicators (51) only; same repeated 5-fold CV as roar.py. usage: python3 roar_mask.py PIPE MODEL"""
import sys, glob, json, random
import numpy as np
sys.path.insert(0, ".")
from kgsim import KG
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
PIPE, MODEL = sys.argv[1], sys.argv[2]
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"])
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
recs = []
for f in sorted(glob.glob(f"rec_{PIPE}_{MODEL}_f*_*.json")): recs += json.load(open(f))["records"]
methods = [k[4:] for k in recs[0] if k.startswith("phi_")] + ["random"]
y = np.array([r["y"] for r in recs])
masks = {(m, k): np.zeros((len(recs), len(A))) for m in methods for k in (1, 2, 3)}
for n, r in enumerate(recs):
    p1, p2 = keys[r["i"]]
    if PIPE == "kg": pl = [i for i in range(len(A)) if kg.ann[p1] & sub[i] and kg.ann[p2] & sub[i]]
    else: pl = sorted({A.index(a) for m in ("D", "LOO") for a in r[f"phi_{m}"]})
    sgn = 1 if r["pred"] == 1 else -1; rr = random.Random(r["i"])
    for m in methods:
        if m == "random": order = rr.sample(pl, len(pl))
        else:
            phi = r[f"phi_{m}"]; tb = {i: rr.random() for i in pl}
            order = sorted(pl, key=lambda i: (-sgn * phi.get(A[i], 0.0), tb[i]))
        for k in (1, 2, 3): masks[(m, k)][n, order[:k]] = 1
splits = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0).split(y, y))
out = {}
for (m, k), M in masks.items():
    acc = [RandomForestClassifier(100, n_jobs=4, random_state=0).fit(M[tr], y[tr]).score(M[te], y[te]) for tr, te in splits]
    out[f"{m}|{k}"] = float(np.mean(acc)); print(m, k, "mask-only acc %.3f" % np.mean(acc), flush=True)
json.dump(out, open(f"roar_mask_{PIPE}_{MODEL}.json", "w"))


Writing roar_mask.py


In [18]:
%%writefile roar_analyze.py
"""Summarise roar_*.json (+ roar_mask_*.json): mean accuracy after retraining, drop vs base, Wilcoxon of KAS (D) vs each
method over the same CV splits (one-sided alternative: D gives LOWER accuracy = more faithful)."""
import json, glob, numpy as np
from scipy.stats import wilcoxon
out = {}
for f in sorted(glob.glob("roar_kg_*.json") + glob.glob("roar_seek_*.json") + glob.glob("roar_hyb_*.json")):
    if "mask" in f: continue
    d = json.load(open(f)); cfg = f"{d['pipe']}|{d['model']}"
    mf = f"roar_mask_{d['pipe']}_{d['model']}.json"; mask = json.load(open(mf)) if glob.glob(mf) else {}
    b = np.mean(d["base"]["acc"]); print(f"\n{cfg}  n={d['n_pairs']}  base acc {b:.3f}")
    out[cfg] = dict(base=b, rows={})
    for key, v in d["res"].items():
        op, m, k = key.split("|"); a = np.array(v["acc"])
        dd = np.array(d["res"][f"{op}|D|{k}"]["acc"])
        p = float(wilcoxon(dd, a, alternative="less").pvalue) if m != "D" and np.any(dd != a) else None
        out[cfg]["rows"][key] = dict(acc=float(a.mean()), drop=float(b - a.mean()), p_D_less=p, mask_acc=mask.get(f"{m}|{k}"))
        print(f"  {op:4s} {m:9s} k={k}  acc {a.mean():.3f}  drop {b-a.mean():.3f}" + (f"  p(D<{m})={p:.2g}" if p is not None else "") +
              (f"  mask-only {mask[f'{m}|{k}']:.3f}" if f"{m}|{k}" in mask else ""))
json.dump(out, open("roar_summary.json", "w"), indent=1)


Writing roar_analyze.py


In [19]:
%%writefile get_msigdb.py
"""Extract Reactome and KEGG gene sets (gene symbols) from MSigDB 7.5.1 as bundled in the conda-forge package r-msigdbr 7.5.1
(R lazy-load database, LZMA-compressed objects). Writes REACTOME_msigdb751.gmt and KEGG_msigdb751.gmt. Needs pyreadr."""
import os, io, lzma, struct, tarfile, urllib.request, tempfile, pyreadr
URL = "https://conda.anaconda.org/conda-forge/noarch/r-msigdbr-7.5.1-r42hc72bb7e_1.tar.bz2"
tmp = tempfile.mkdtemp()
tb = os.path.join(tmp, "m.tar.bz2"); urllib.request.urlretrieve(URL, tb)
with tarfile.open(tb) as t:
    b = t.extractfile("lib/R/library/msigdbr/R/sysdata.rdb").read()
objs = []; off = 0
while off < len(b):
    dec = lzma.LZMADecompressor(format=lzma.FORMAT_RAW, filters=[{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}])
    d = dec.decompress(b[off + 5:]); used = len(b) - off - 5 - len(dec.unused_data)
    p = os.path.join(tmp, f"o{len(objs)}.rds"); open(p, "wb").write(d); objs.append(pyreadr.read_r(p)[None]); off += 5 + used
genes, gsg, gs = objs  # msigdbr_genes, msigdbr_geneset_genes, msigdbr_genesets
for sub in ["CP:REACTOME", "CP:KEGG"]:
    s = gs[gs.gs_subcat == sub]; m = gsg[gsg.gs_id.isin(s.gs_id)].merge(genes, on="gene_id"); name = s.set_index("gs_id")["gs_name"]
    with open(sub.split(":")[1] + "_msigdb751.gmt", "w") as f:
        for gid, grp in m.groupby("gs_id"):
            f.write(name[gid] + "\t" + gid + "\t" + "\t".join(sorted(set(grp.human_gene_symbol))) + "\n")
    print(sub, len(s), "gene sets")


Writing get_msigdb.py


In [20]:
%%writefile plaus.py
"""Quantitative Plausibility proxy: does an explanation's top aspect agree with external pathway knowledge?
usage: python3 plaus.py GMT_NAME     (reads {GMT_NAME}_msigdb751.gmt: REACTOME / KEGG from MSigDB 7.5.1, or REACTOME_LIVE
       written by get_reactome_live.py from the current reactome.org UniProt2Reactome file)
For every explained pair with y=1 and pred=1 (all rec_*_{rf,xgb}_f*.json), take the smallest gene set (10..300 members in the
GAF universe) that contains both proteins. Reference relevance of an aspect = over-representation (one-sided hypergeometric,
BH over the 51 aspects) of the OTHER members of that pathway (the two proteins excluded) among proteins annotated under the
aspect, versus all GAF proteins. Annotations used for the reference exclude evidence IEA and anything assigned_by Reactome,
so the reference does not reuse Reactome-derived GO annotations.
A pair is informative when it has >=2 players and at least one relevant and one non-relevant player.
Metrics over the pair's players: hit@1, precision@3, NDCG@3 (gain = -log10 p, capped at 10).
Baselines: random order (exact expectation for hit@1, 200 permutations otherwise) and, for KGsim2vec, ranking players by
their raw similarity feature (model-free)."""
import sys, glob, json, math, random, collections
import numpy as np
from scipy.stats import hypergeom, wilcoxon
sys.path.insert(0, ".")
from kgsim import KG

GMT = sys.argv[1]
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"])
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
Xpy = K["Xpy"][:, keep]

# filtered annotations (no IEA, not assigned by Reactome) and symbol -> accession
filt = collections.defaultdict(set); sym2acc = collections.defaultdict(set)
for l in open("godata/goa_human.gaf"):
    if l.startswith("!"): continue
    f = l.rstrip("\n").split("\t")
    if len(f) < 15: continue
    sym2acc[f[2]].add(f[1])
    if "NOT" in f[3] or f[6] == "IEA" or f[14] == "Reactome": continue
    filt[f[1]].add("GO_" + f[4].split(":")[1])
U = sorted(filt); NU = len(U)
asp_has = np.array([[bool(filt[p] & sub[a]) for a in range(len(A))] for p in U])  # NU x 51
uidx = {p: i for i, p in enumerate(U)}
bg = asp_has.sum(0)

sets = []
for l in open(f"{GMT}_msigdb751.gmt"):
    f = l.rstrip("\n").split("\t")
    mem = {a for s in f[2:] for a in (sym2acc.get(s) or {s})} & set(uidx)   # members may be gene symbols or UniProt accessions
    if 10 <= len(mem) <= 300: sets.append((f[0], mem))
member_of = collections.defaultdict(list)
for j, (_, mem) in enumerate(sets):
    for p in mem: member_of[p].append(j)
print(GMT, "gene sets kept", len(sets), "universe", NU, flush=True)

relcache = {}
def relevance(j, p1, p2):
    key = (j, p1, p2)
    if key in relcache: return relcache[key]
    M = [uidx[p] for p in sets[j][1] if p not in (p1, p2)]
    cnt = asp_has[M].sum(0); n = len(M)
    pv = np.array([hypergeom.sf(cnt[a] - 1, NU, bg[a], n) if bg[a] > 0 else 1.0 for a in range(len(A))])
    order = np.argsort(pv); q = np.empty_like(pv); m = len(pv); prev = 1.0
    for rank, a in reversed(list(enumerate(order))):
        prev = min(prev, pv[a] * m / (rank + 1)); q[a] = prev
    relcache[key] = (q < 0.05, np.minimum(-np.log10(np.maximum(pv, 1e-300)), 10.0))
    return relcache[key]

def players(p1, p2):
    t1, t2 = kg.ann[p1], kg.ann[p2]
    return [i for i in range(len(A)) if t1 & sub[i] and t2 & sub[i]]
def ndcg3(order, gain):
    dcg = sum(gain[a] / math.log2(r + 2) for r, a in enumerate(order[:3]))
    ideal = sorted((gain[a] for a in order), reverse=True)
    idcg = sum(g / math.log2(r + 2) for r, g in enumerate(ideal[:3]))
    return dcg / idcg if idcg > 0 else 0.0

out = {}; cases = []
for cfg in ["kg|rf", "kg|xgb", "seek|rf", "seek|xgb", "hyb|rf", "hyb|xgb"]:
    pipe, model = cfg.split("|"); recs = []
    for fn in sorted(glob.glob(f"rec_{pipe}_{model}_f*_*.json")): recs += json.load(open(fn))["records"]
    methods = [k[4:] for k in recs[0] if k.startswith("phi_")]
    res = collections.defaultdict(lambda: collections.defaultdict(list)); n_tp = n_shared = 0
    for r in recs:
        if not (r["y"] == 1 and r["pred"] == 1): continue
        n_tp += 1
        p1, p2 = keys[r["i"]]
        shared = set(member_of.get(p1, ())) & set(member_of.get(p2, ()))
        if not shared: continue
        n_shared += 1
        j = min(shared, key=lambda j: len(sets[j][1]))
        rel, gain = relevance(j, p1, p2)
        # players: KGsim2vec players for kg; for seek/hyb, aspects with non-zero KAS or LOO attribution (= SEEK players)
        pl = players(p1, p2) if pipe == "kg" else sorted({A.index(a) for m in ("D", "LOO") for a in r[f"phi_{m}"]})
        nrel = int(sum(rel[i] for i in pl))
        if len(pl) < 2 or nrel == 0 or nrel == len(pl): continue
        rr = random.Random(r["i"])
        orders = {}
        for m in methods:
            phi = r[f"phi_{m}"]; tb = {i: rr.random() for i in pl}
            orders[m] = sorted(pl, key=lambda i: (-phi.get(A[i], 0.0), tb[i]))
        if pipe == "kg":
            tb = {i: rr.random() for i in pl}
            orders["Sim"] = sorted(pl, key=lambda i: (-Xpy[r["i"], i], tb[i]))
        for m, o in orders.items():
            res[m]["hit1"].append(float(rel[o[0]])); res[m]["p3"].append(float(np.mean([rel[a] for a in o[:3]])))
            res[m]["ndcg3"].append(ndcg3(o, gain))
        res["random"]["hit1"].append(nrel / len(pl))
        perms = [rr.sample(pl, len(pl)) for _ in range(200)]
        res["random"]["p3"].append(float(np.mean([np.mean([rel[a] for a in o[:3]]) for o in perms])))
        res["random"]["ndcg3"].append(float(np.mean([ndcg3(o, gain) for o in perms])))
        if cfg == "kg|rf":
            cases.append(dict(pair=r["pair"], pathway=sets[j][0], size=len(sets[j][1]), players=[kg.label.get(A[i], A[i]) for i in pl],
                              relevant=[kg.label.get(A[i], A[i]) for i in pl if rel[i]],
                              top3={m: [kg.label.get(A[i], A[i]) for i in o[:3]] for m, o in orders.items()}))
    summ = dict(n_tp=n_tp, n_shared=n_shared, n_informative=len(res["random"]["hit1"]), metrics={}, wilcoxon_vs_D={})
    for m, d in res.items():
        summ["metrics"][m] = {k: dict(mean=float(np.mean(v)),
                                      ci95=[float(x) for x in np.percentile([np.mean(np.random.default_rng(b).choice(v, len(v))) for b in range(1000)], [2.5, 97.5])])
                              for k, v in d.items()}
    for m in res:
        if m == "D": continue
        for k in ("hit1", "ndcg3"):
            a, b = np.array(res["D"][k]), np.array(res[m][k])
            summ["wilcoxon_vs_D"][f"{m}|{k}"] = float(wilcoxon(a, b).pvalue) if np.any(a != b) else 1.0
    out[cfg] = summ
    print(cfg, "TP", n_tp, "shared", n_shared, "informative", summ["n_informative"])
    for m in summ["metrics"]:
        mm = summ["metrics"][m]
        print("   %-16s hit@1 %.3f  p@3 %.3f  ndcg@3 %.3f" % (m, mm["hit1"]["mean"], mm["p3"]["mean"], mm["ndcg3"]["mean"]),
              "" if m == "D" else "p(hit1)=%.2g p(ndcg)=%.2g" % (summ["wilcoxon_vs_D"].get(f"{m}|hit1", 1), summ["wilcoxon_vs_D"].get(f"{m}|ndcg3", 1)))
json.dump(dict(gmt=GMT, results=out, cases=cases), open(f"plaus_{GMT}.json", "w"))


Writing plaus.py


In [21]:
%%writefile case_study.py
"""Plausibility case study table (KGsim2vec + RF, records from the 10-fold runs): top-3 aspects of KAS / TreeSHAP / LIME
vs the aspects over-represented in the pair's shared Reactome pathway (plaus.py reference). Output: case_study.md"""
import json, collections
d = json.load(open("plaus_REACTOME.json")); cases = {c["pair"]: c for c in d["cases"]}
sym = {}
for l in open("godata/goa_human.gaf"):
    if l.startswith("!"): continue
    f = l.split("\t"); sym.setdefault(f[1], f[2])
PAIRS = ["P00533-P15514", "Q05086-Q16763", "P62917-P05388", "Q9BYM8-P21580", "P19447-P20226", "Q9Y6K5-P29728",
         "O60231-Q7L014", "Q9Y5J6-O14925", "Q15788-Q13133", "Q9UJX6-O95997", "Q9Y676-P82650", "Q93034-P46934"]
def fmt(lst, rel): return "; ".join(("**%s** ✓" % a) if a in rel else a for a in lst)
L = ["| Cặp (gene) | Pathway Reactome chung (số gene) | Aspect được làm giàu | KAS top-3 | TreeSHAP top-3 | LIME top-3 |", "|---|---|---|---|---|---|"]
hits = collections.Counter()
for p in PAIRS:
    c = cases[p]; a, b = p.split("-"); rel = set(c["relevant"])
    L.append(f"| {sym.get(a,a)}–{sym.get(b,b)} | {c['pathway'].replace('REACTOME_','').replace('_',' ').lower()} ({c['size']}) | {', '.join(c['relevant'])} | "
             f"{fmt(c['top3']['D'],rel)} | {fmt(c['top3']['TreeSHAP'],rel)} | {fmt(c['top3']['LIME'],rel)} |")
    for m in ("D", "TreeSHAP", "LIME"): hits[m] += c["top3"][m][0] in rel
L.append(""); L.append("Top-1 nằm trong aspect được làm giàu: " + ", ".join(f"{'KAS' if m=='D' else m} {hits[m]}/{len(PAIRS)}" for m in ("D", "TreeSHAP", "LIME")))
open("case_study.md", "w").write("\n".join(L) + "\n"); print("\n".join(L))


Writing case_study.py


In [22]:
# !cp records/rec_*.json .   # bỏ comment nếu dùng records có sẵn
!pip -q install pyreadr
for c in ['kg rf','kg xgb','seek rf','seek xgb']:
    !python roar.py {c} > log_roar_{c.replace(' ','_')}.txt 2>&1
    !python roar_mask.py {c}
!python roar_analyze.py

D 1 mask-only acc 0.697
D 2 mask-only acc 0.743
D 3 mask-only acc 0.767
LOO 1 mask-only acc 0.687
LOO 2 mask-only acc 0.557
LOO 3 mask-only acc 0.663
FS0 1 mask-only acc 0.633
FS0 2 mask-only acc 0.693
FS0 3 mask-only acc 0.640
TreeSHAP 1 mask-only acc 0.640
TreeSHAP 2 mask-only acc 0.720
TreeSHAP 3 mask-only acc 0.563
LIME 1 mask-only acc 0.710
LIME 2 mask-only acc 0.713
LIME 3 mask-only acc 0.630
random 1 mask-only acc 0.540
random 2 mask-only acc 0.547
random 3 mask-only acc 0.590
D 1 mask-only acc 0.670
D 2 mask-only acc 0.757
D 3 mask-only acc 0.753
LOO 1 mask-only acc 0.637
LOO 2 mask-only acc 0.653
LOO 3 mask-only acc 0.670
FS0 1 mask-only acc 0.820
FS0 2 mask-only acc 0.823
FS0 3 mask-only acc 0.750
TreeSHAP 1 mask-only acc 0.683
TreeSHAP 2 mask-only acc 0.817
TreeSHAP 3 mask-only acc 0.640
LIME 1 mask-only acc 0.690
LIME 2 mask-only acc 0.727
LIME 3 mask-only acc 0.627
random 1 mask-only acc 0.540
random 2 mask-only acc 0.547
random 3 mask-only acc 0.590
D 1 mask-only acc 0.65

In [23]:
!python get_msigdb.py
!python plaus.py REACTOME
!python plaus.py KEGG
!python case_study.py

CP:REACTOME 1615 gene sets
CP:KEGG 186 gene sets
REACTOME gene sets kept 1253 universe 18722
kg|rf TP 46 shared 38 informative 34
   D                hit@1 0.706  p@3 0.578  ndcg@3 0.704 
   LOO              hit@1 0.735  p@3 0.569  ndcg@3 0.670 p(hit1)=0.65 p(ndcg)=0.3
   FS0              hit@1 0.706  p@3 0.569  ndcg@3 0.680 p(hit1)=1 p(ndcg)=0.068
   TreeSHAP         hit@1 0.765  p@3 0.559  ndcg@3 0.702 p(hit1)=0.32 p(ndcg)=0.73
   LIME             hit@1 0.588  p@3 0.500  ndcg@3 0.596 p(hit1)=0.21 p(ndcg)=0.027
   Sim              hit@1 0.618  p@3 0.520  ndcg@3 0.625 p(hit1)=0.44 p(ndcg)=0.089
   random           hit@1 0.495  p@3 0.497  ndcg@3 0.567 p(hit1)=0.012 p(ndcg)=0.00097
kg|xgb TP 46 shared 37 informative 33
   D                hit@1 0.758  p@3 0.626  ndcg@3 0.739 
   LOO              hit@1 0.636  p@3 0.586  ndcg@3 0.678 p(hit1)=0.046 p(ndcg)=0.02
   FS0              hit@1 0.545  p@3 0.596  ndcg@3 0.696 p(hit1)=0.02 p(ndcg)=0.34
   TreeSHAP         hit@1 0.667  p@3 0.586  ndcg

## 6. ROAR đầy đủ trên 47,142 cặp (KGsim2vec + RF)
Chỉ cần chạy mục cài đặt, mục Mã nguồn và mục 1 (tạo `feats_kg.npz`). Không cần mục 3–5.
1. `explain_all.py`: giải thích **mọi** cặp bằng KAS, LOO, FS0, TreeSHAP, LIME. Mỗi cặp do mô hình của fold mà nó là cặp test giải thích. Mỗi fold chia thành nhiều phần để dùng hết CPU; phần đã có file sẽ được bỏ qua, nên nếu Colab ngắt thì chạy lại ô này là chạy tiếp.
2. `roar_full.py`: bỏ top-k aspect (k = 1, 2, 3) của từng cặp, tính lại đặc trưng, huấn luyện lại RF trên 10 partition gốc, báo median WAF và mức giảm; kèm kiểm tra rò rỉ mask và Wilcoxon (KAS so với từng phương pháp, 10 fold).

**Nên dùng runtime CPU High-RAM** (không cần GPU). Nên lưu kết quả sang Google Drive vì phiên Colab có thể ngắt.

Ước tính với 8 vCPU: giải thích 1.5–2 giờ, huấn luyện lại 20–30 phút.

In [24]:
%%writefile explain_all.py
"""Explain EVERY test pair of one fold (KGsim2vec view) with KAS, LOO, FS0, TreeSHAP, LIME, for full-data ROAR.
usage: python3 explain_all.py MODEL FOLD [PART NPARTS]   -> all_kg_{MODEL}_f{FOLD}[_p{PART}].json
(each pair explained by the fold model that did not train on it; PART/NPARTS split a fold across CPUs)"""
import sys, json, time
import numpy as np

# Vá tương thích toàn diện cho shap <= 0.41 trên NumPy mới
for attr, target in [
    ("bool", bool),
    ("bool8", np.bool_),
    ("int", int),
    ("float", float),
]:
  if not hasattr(np, attr):
    setattr(np, attr, target)

MODEL, FOLD = sys.argv[1], int(sys.argv[2])
PART, NPARTS = (int(sys.argv[3]), int(sys.argv[4])) if len(sys.argv) > 4 else (0, 1)
sys.argv = ["x", "kg", MODEL, str(FOLD), "1"]
exec(open("explain_v2.py").read().split("pos=[i for i in test")[0])   # setup: data, model, feats, explainers
out = []; t0 = time.time()
for n, i in enumerate(test[PART::NPARTS]):
    p1, p2 = keys[i]
    st = (set(kg.ann[p1]), set(kg.ann[p2]), set(), set())
    p = proba(feats(st))[0]; r = dict(i=int(i), y=int(y[i]), pred=int(p > 0.5))
    for name, f in METHODS.items():
        phi = f(st, 0); r[f"phi_{name}"] = {A[j]: round(float(phi[j]), 5) for j in np.nonzero(phi)[0]}
    out.append(r)
    if n % 500 == 0: print(FOLD, n, round(time.time() - t0), "s", flush=True)
json.dump(dict(model=MODEL, fold=FOLD, aspects=A, records=out), open(f"all_kg_{MODEL}_f{FOLD}" + (f"_p{PART}" if NPARTS > 1 else "") + ".json", "w"))
print("done", FOLD, time.time() - t0)


Writing explain_all.py


In [25]:
%%writefile roar_full.py
"""Full-data ROAR for KGsim2vec: every one of the 47,142 pairs was explained (explain_all.py) by the fold model that did not
train on it. Each pair loses its own top-k players (knowledge-space removal, recompute), then the model is retrained and
evaluated on the repo's 10 partitions (train on 9, test on the held-out partition), exactly like the prediction table.
Also the mask-only leakage check (RF on removed-aspect indicators). usage: python3 roar_full.py MODEL"""
import sys, glob, json, random, time
import numpy as np
sys.path.insert(0, "."); sys.path.insert(0, "ExplainablePPI")
from kgsim import KG
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score
from scipy.stats import wilcoxon
MODEL = sys.argv[1]; KS = [1, 2, 3]
K = np.load("feats_kg.npz"); keys = [tuple(k) for k in K["keys"]]; aspects = list(K["aspects"]); y = K["y"]
ci = aspects.index("GO_0032991"); keep = [j for j in range(52) if j != ci]; A = [aspects[j] for j in keep]
kg = KG("godata/go-basic.owl", "godata/goa_human.gaf", aspects); sub = [kg.sub[j] for j in keep]
X0 = K["Xpy"][:, keep]
rec = {}
for f in sorted(glob.glob(f"all_kg_{MODEL}_f*.json")):
    for r in json.load(open(f))["records"]: rec[r["i"]] = r
assert len(rec) == len(y), (len(rec), len(y))
methods = [k[4:] for k in rec[0] if k.startswith("phi_")] + ["random"]
def feats(t1, t2): v = kg.features(t1, t2); return [v[j] for j in keep]
Xs = {(m, k): np.zeros_like(X0) for m in methods for k in KS}; Ms = {(m, k): np.zeros_like(X0) for m in methods for k in KS}
t0 = time.time()
for i in range(len(y)):
    r = rec[i]; p1, p2 = keys[i]; t1, t2 = kg.ann[p1], kg.ann[p2]
    pl = [a for a in range(len(A)) if t1 & sub[a] and t2 & sub[a]]
    sgn = 1 if r["pred"] == 1 else -1; rr = random.Random(i)
    for m in methods:
        if m == "random": order = rr.sample(pl, len(pl))
        else:
            phi = r[f"phi_{m}"]; tb = {a: rr.random() for a in pl}
            order = sorted(pl, key=lambda a: (-sgn * phi.get(A[a], 0.0), tb[a]))
        for k in KS:
            S = order[:k]
            if S:
                U = set().union(*[sub[a] for a in S]); Xs[(m, k)][i] = feats(t1 - U, t2 - U); Ms[(m, k)][i, S] = 1
            else: Xs[(m, k)][i] = X0[i]
    if i % 5000 == 0: print(i, round(time.time() - t0), "s", flush=True)
P = "ExplainablePPI/Experimental_Results/Data/StratifiedPartitions/Indexes__crossvalidationTest__Run"
folds = [[int(l) for l in open(P + ("1(complete)" if f == 1 else str(f)) + ".txt") if l.strip()] for f in range(1, 11)]
def make(): return (RandomForestClassifier(100, n_jobs=4, random_state=0) if MODEL == "rf"
                    else XGBClassifier(n_estimators=300, n_jobs=4, random_state=0, eval_metric="logloss"))
def cv(X, mk=make):
    f1s = []
    for te in folds:
        m = np.zeros(len(y), bool); m[te] = True
        c = mk().fit(X[~m], y[~m]); f1s.append(f1_score(y[m], c.predict(X[m]), average="weighted"))
    return f1s
out = dict(model=MODEL, n=len(y), base=cv(X0), res={}, mask={})
print("base WAF %.4f" % np.median(out["base"]), flush=True)
for m in methods:
    for k in KS:
        out["res"][f"{m}|{k}"] = cv(Xs[(m, k)])
        out["mask"][f"{m}|{k}"] = float(np.mean(cv(Ms[(m, k)], lambda: RandomForestClassifier(100, n_jobs=4, random_state=0))))
        a = np.array(out["res"][f"{m}|{k}"])
        print(m, k, "WAF median %.4f drop %.4f mask-only %.3f" % (np.median(a), np.median(out["base"]) - np.median(a), out["mask"][f"{m}|{k}"]), round(time.time() - t0), "s", flush=True)
for k in KS:
    d = np.array(out["res"][f"D|{k}"])
    for m in methods:
        if m != "D":
            o = np.array(out["res"][f"{m}|{k}"]); out[f"p_D_less_{m}_{k}"] = float(wilcoxon(d, o, alternative="less").pvalue)
            print(f"k={k} p(D<{m}) = {out[f'p_D_less_{m}_{k}']:.3g}")
json.dump(out, open(f"roar_full_kg_{MODEL}.json", "w"))


Writing roar_full.py


In [26]:
import os, subprocess, time
from concurrent.futures import ThreadPoolExecutor
MODEL = 'rf'
NCPU = os.cpu_count(); NPARTS = max(1, NCPU // 2)   # 10 fold x NPARTS phần
print('CPU:', NCPU, '| mỗi fold chia', NPARTS, 'phần')
jobs = [(f, p) for f in range(1, 11) for p in range(NPARTS)]
def run(j):
    f, p = j; out = f'all_kg_{MODEL}_f{f}_p{p}.json' if NPARTS > 1 else f'all_kg_{MODEL}_f{f}.json'
    if os.path.exists(out): return out + ': đã có'
    t = time.time(); r = subprocess.run(['python', 'explain_all.py', MODEL, str(f), str(p), str(NPARTS)], capture_output=True, text=True)
    return out + (f': xong sau {(time.time()-t)/60:.1f} phút' if r.returncode == 0 else ': LỖI\n' + r.stderr[-1500:])
t0 = time.time()
with ThreadPoolExecutor(max_workers=NCPU) as ex:
    for i, msg in enumerate(ex.map(run, jobs)): print(f'[{i+1}/{len(jobs)}, {(time.time()-t0)/60:.0f} phút]', msg, flush=True)

CPU: 8 | mỗi fold chia 4 phần
[1/40, 14 phút] all_kg_rf_f1_p0.json: xong sau 14.0 phút
[2/40, 14 phút] all_kg_rf_f1_p1.json: xong sau 14.2 phút
[3/40, 14 phút] all_kg_rf_f1_p2.json: xong sau 13.4 phút
[4/40, 14 phút] all_kg_rf_f1_p3.json: xong sau 13.0 phút
[5/40, 14 phút] all_kg_rf_f2_p0.json: xong sau 13.2 phút
[6/40, 14 phút] all_kg_rf_f2_p1.json: xong sau 13.6 phút
[7/40, 14 phút] all_kg_rf_f2_p2.json: xong sau 13.0 phút
[8/40, 14 phút] all_kg_rf_f2_p3.json: xong sau 13.3 phút
[9/40, 26 phút] all_kg_rf_f3_p0.json: xong sau 13.4 phút
[10/40, 26 phút] all_kg_rf_f3_p1.json: xong sau 12.5 phút
[11/40, 26 phút] all_kg_rf_f3_p2.json: xong sau 12.9 phút
[12/40, 26 phút] all_kg_rf_f3_p3.json: xong sau 12.8 phút
[13/40, 26 phút] all_kg_rf_f4_p0.json: xong sau 12.3 phút
[14/40, 27 phút] all_kg_rf_f4_p1.json: xong sau 13.2 phút
[15/40, 28 phút] all_kg_rf_f4_p2.json: xong sau 13.7 phút
[16/40, 28 phút] all_kg_rf_f4_p3.json: xong sau 12.9 phút
[17/40, 37 phút] all_kg_rf_f5_p0.json: xong sau 11.

In [27]:
!python roar_full.py rf | tee roar_full_rf.txt
# Lưu kết quả sang Drive (khuyến nghị):
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/KAS_ROAR && cp all_kg_rf_f*.json roar_full_kg_rf.json roar_full_rf.txt /content/drive/MyDrive/KAS_ROAR/

/usr/local/lib/python3.13/dist-packages/xgboost/compat.py:106: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
0 0 s
5000 43 s
10000 85 s
15000 128 s
20000 172 s
25000 216 s
30000 258 s
35000 301 s
40000 343 s
45000 385 s
base WAF 0.9246
D 1 WAF median 0.8446 drop 0.0800 mask-only 0.682 427 s
D 2 WAF median 0.8117 drop 0.1129 mask-only 0.712 442 s
D 3 WAF median 0.7928 drop 0.1318 mask-only 0.787 456 s
LOO 1 WAF median 0.8410 drop 0.0836 mask-only 0.593 471 s
LOO 2 WAF median 0.7950 drop 0.1296 mask-only 0.618 486 s
LOO 3 WAF median 0.7245 drop 0.2001 mask-only 0.662 501 s
FS0 1 WAF median 0.8400 drop 0.0845 mask-only 0.658 516 s
FS0 2 WAF median 0.7955 drop 0.1290 mask-only 0.685 533 s
FS0 3 WAF median 0.7406 drop 0.1840 mask-only 0.693 548 s
TreeSHAP 1 WAF median 0.8345 

## 7. Plausibility với Reactome bản mới nhất (reactome.org)
Tải `UniProt2Reactome_All_Levels.txt` hiện hành (người, mọi mức pathway), rồi chạy cùng thước đo như mục 5 để so với kết quả dùng MSigDB 7.5.1. Cần `rec_*.json` của MODE `'full'` (hoặc thư mục `records/` có sẵn).

In [28]:
%%writefile get_reactome_live.py
"""Current Reactome release straight from reactome.org (human, all pathway levels, UniProt accessions).
Writes REACTOME_LIVE_msigdb751.gmt (same format as the MSigDB files; name kept so plaus.py can read it)."""
import urllib.request, collections
URL = "https://reactome.org/download/current/UniProt2Reactome_All_Levels.txt"
sets = collections.defaultdict(set); names = {}
for line in urllib.request.urlopen(URL).read().decode().splitlines():
    f = line.split("\t")
    if len(f) < 6 or f[5] != "Homo sapiens": continue
    acc = f[0].split("-")[0]; sets[f[1]].add(acc); names[f[1]] = f[3]
with open("REACTOME_LIVE_msigdb751.gmt", "w") as out:
    for pid, mem in sets.items():
        out.write(("REACTOME_LIVE_" + names[pid].upper().replace(" ", "_"))[:200] + "\t" + pid + "\t" + "\t".join(sorted(mem)) + "\n")
print("human pathways", len(sets))


Writing get_reactome_live.py


In [29]:
!python get_reactome_live.py
!python plaus.py REACTOME_LIVE | tee plaus_REACTOME_LIVE.txt

Traceback (most recent call last):
  File "/content/get_reactome_live.py", line 6, in <module>
    for line in urllib.request.urlopen(URL).read().decode().splitlines():
                ~~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 495, in open
    response = meth(req, response)
  File "/usr/lib/python3.13/urllib/request.py", line 604, in http_response
    response = self.parent.error(
        'http', request, response, code, msg, hdrs)
  File "/usr/lib/python3.13/urllib/request.py", line 533, in error
    return self._call_chain(*args)
           ~~~~~~~~~~~~~~~~^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 466, in _call_chain
    result = func(*args)
  File "/usr/lib/python3.13/urllib/request.py", line 613, in http_error_default
    raise HTTPError(req.full_url, code, msg, hdrs, fp)


In [30]:
!pip freeze > pip_freeze_colab.txt